<a href="https://colab.research.google.com/github/sahlulfurqon-coder/ComicDownloader-SuwayomiServer/blob/main/comicdownloader_stable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Project ini dibuat untuk mengunduh resources manga per chapter dari project github yang bernama "**Suwayomi**" dengan bantuan **Google Drive** sebagai penyimpanan hasil download dan Cloudflare sebagai tunnel servernya, disini digunakan flaresolver untuk menembus perlindungan anti-bot dari Cloudflare dan DDoS-GUARD

(Gunakan dengan Bijak)

Note: setelah berhasil menjalankan server cloudflare untuk membuka Suwayomi, klik browse dan tambahkan extension berikut

https://raw.githubusercontent.com/keiyoushi/extensions/repo/index.json

In [ ]:
import os
import re
import json
import time
import urllib.request
import subprocess
from google.colab import drive

# 1. Sambungkan Google Drive & Set Symlink
print("📂 Menyambungkan Google Drive...")
drive.mount('/content/drive', force_remount=True)

drive_folder = "/content/drive/MyDrive/SuwayomiData"
os.makedirs(f"{drive_folder}/downloads", exist_ok=True)

os.makedirs("/root/.local/share", exist_ok=True)
subprocess.run("rm -rf /root/.local/share/Tachidesk /root/.local/share/Suwayomi", shell=True)
subprocess.run(f"ln -s '{drive_folder}' /root/.local/share/Tachidesk", shell=True)
subprocess.run(f"ln -s '{drive_folder}' /root/.local/share/Suwayomi", shell=True)

print("✅ Google Drive terhubung!")

# 2. Install Dependensi (Java 21, Chromium, Xvfb)
print("📦 Memasang dependensi sistem...")
subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-21-jre-headless chromium-browser chromium-chromedriver xvfb libgl1-mesa-glx libglib2.0-0 > /dev/null", shell=True)

# 3. FlareSolverr
if not os.path.exists("flaresolverr"):
    print("🛡️ Mengunduh FlareSolverr...")
    flaresolverr_url = "https://github.com/FlareSolverr/FlareSolverr/releases/latest/download/flaresolverr_linux_x64.tar.gz"
    subprocess.run(f"wget -q '{flaresolverr_url}' -O flaresolverr.tar.gz && tar -xzf flaresolverr.tar.gz", shell=True)

print("🚀 Memulai FlareSolverr...")
flaresolverr_log = open("flaresolverr.log", "w")
subprocess.Popen(["xvfb-run", "--auto-servernum", "./flaresolverr/flaresolverr"], stdout=flaresolverr_log, stderr=subprocess.STDOUT)
time.sleep(6)

# 4. Unduh Binary jika belum ada
if not os.path.exists("suwayomi.jar"):
    api_url = "https://api.github.com/repos/Suwayomi/Suwayomi-Server/releases/latest"
    try:
        req = urllib.request.Request(api_url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read().decode())
            jar_url = next(asset["browser_download_url"] for asset in data["assets"] if asset["name"].endswith(".jar"))
            subprocess.run(f"wget -q '{jar_url}' -O suwayomi.jar", shell=True)
    except Exception as e:
        print(f"❌ Error Suwayomi Jar: {e}")

if not os.path.exists("cloudflared"):
    subprocess.run("wget -q 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64' -O cloudflared && chmod +x cloudflared", shell=True)

# 5. Jalankan Suwayomi Server
print("🚀 Memulai Suwayomi Server...")
log_file = open("suwayomi.log", "w")
suwayomi_process = subprocess.Popen(["java", "-Djava.awt.headless=true", "-jar", "suwayomi.jar", "--server.port=4567"], stdout=log_file, stderr=subprocess.STDOUT)

# 6. Tunggu Siap & Buat Cloudflare Tunnel
print("⏳ Menunggu server siap...")
server_ready = False
for _ in range(40):
    if suwayomi_process.poll() is not None:
        print("❌ Suwayomi Crash!")
        break
    try:
        with urllib.request.urlopen("http://localhost:4567") as response:
            if response.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(2)

if server_ready:
    print("✅ Suwayomi Siap!")
    tunnel = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:4567"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in iter(tunnel.stdout.readline, ""):
        if "trycloudflare.com" in line:
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
            if match:
                print("\n==================================================")
                print("🎉 BUKA SUWAYOMI DI BROWSER ANDA:")
                print(f"👉 {match.group(0)}")
                print("==================================================")
                break